In [ ]:
# ---------------------------------
# Read a single subject's interpolated.csv file safely
# ---------------------------------
def read_interpolated_sensor_csv(subject_path: Path):
    interpolated_files = list(subject_path.glob("interpolated.csv"))
    if not interpolated_files:
        return None  # No interpolated.csv found
    df = pd.read_csv(interpolated_files[0])

    required_cols = ["left finger", "right finger", "left wrist", "right wrist"]
    for col in required_cols:
        if col not in df.columns:
            return None  # Missing required column

    return {
        "left finger": df["left finger"],
        "right finger": df["right finger"],
        "left wrist": df["left wrist"],
        "right wrist": df["right wrist"]
    }

# ---------------------------------
# Assign affected vs less affected per subject with safety checks
# ---------------------------------
def assign_affected_and_less_affected(interpolated_dir: Path, output_dir: Path):
    """
    Reads each subject's interpolated.csv and demographics CSV
    Creates assigned.csv with affected/less affected columns.
    Skips subjects with missing data or files.
    """

    df_demo = pd.read_csv("path/to/clinical_scores.csv")

    subject_paths = get_subject_paths(interpolated_dir)
    create_output_dirs(subject_paths, output_dir)

    for subject_path in subject_paths:
        subject = subject_path.name
        print(f" Checking subject: {subject}")

        # Read interpolated data
        interpolated_dict = read_interpolated_sensor_csv(subject_path)
        if interpolated_dict is None:
            print(f" Skipping {subject}: missing interpolated.csv or required columns")
            continue

        # Lookup affected side
        if subject not in df_demo["SubjID"].values:
            print(f" Skipping {subject}: not found in demographics CSV")
            continue

        affected_side = df_demo.loc[df_demo["SubjID"] == subject, "Affected Side"].iloc[0].strip().lower()

        # Assign affected and less affected sides
        if affected_side == "left":
            assigned_dict = {
                "affected finger": interpolated_dict["left finger"],
                "affected wrist": interpolated_dict["left wrist"],
                "less affected finger": interpolated_dict["right finger"],
                "less affected wrist": interpolated_dict["right wrist"],
            }
        elif affected_side == "right":
            assigned_dict = {
                "affected finger": interpolated_dict["right finger"],
                "affected wrist": interpolated_dict["right wrist"],
                "less affected finger": interpolated_dict["left finger"],
                "less affected wrist": interpolated_dict["left wrist"],
            }
        else:
            print(f" Invalid affected side for {subject}: {affected_side}")
            continue

        # Compute difference features
        df_assigned = pd.DataFrame({
            "affected finger": assigned_dict["affected finger"],
            "affected wrist": assigned_dict["affected wrist"],
            "less affected finger": assigned_dict["less affected finger"],
            "less affected wrist": assigned_dict["less affected wrist"],
            "affected finger - wrist": assigned_dict["affected finger"] - assigned_dict["affected wrist"],
            "less affected finger - wrist": assigned_dict["less affected finger"] - assigned_dict["less affected wrist"],
        })

        # Save to output directory
        output_file = output_dir / subject / "assigned.csv"
        df_assigned.to_csv(output_file, index=False)
        print(f"Saved: {output_file}")

# ---------------------------------
# Example usage
# ---------------------------------
assign_affected_and_less_affected(
    interpolated_dir=Path("path/to/your/data"),
    output_dir=Path("path/to/your/data")
)
